# 🏪 Entrenamiento Optimizado de YOLOv8 para Tiendita IA

Este notebook entrena el modelo de detección de productos con **GPU en Google Colab (Tesla T4)**.

### 🎯 Clases a Detectar:
1. `Aceite`
2. `Atun`
3. `Leche`
4. `Refresco`
5. `Sopa`
6. `Yogurt`

### ⚙️ Instrucciones Previas:
- Menú superior `Entorno de ejecución` -> `Cambiar tipo de entorno de ejecución` -> Selecciona **GPU T4** -> Guardar.

## 1️⃣ Conectar con Google Drive
Monta tu Google Drive para leer `dataset.zip` y guardar automáticamente los modelos entrenados.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/proyecto_yolo'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/modelos_finales', exist_ok=True)

print(f"✅ Google Drive conectado.")
print(f"📁 Carpeta en Drive: {DRIVE_DIR}")

## 2️⃣ Descomprimir Dataset y Configurar data.yaml Automáticamente (Auto-Normalizado)
Esta celda normaliza automáticamente los archivos creados en Windows para que Linux (Colab) cree las carpetas correctamente.

In [ ]:
import os
import zipfile
import shutil
import glob
import yaml

DATASET_DIR = '/content/dataset'

# 1. Buscar cualquier archivo zip del dataset
candidatos_zip = [
    f'{DRIVE_DIR}/dataset.zip',
    '/content/dataset.zip',
    '/content/ProyectoFinalIA.v5i.yolov8.zip',
    f'{DRIVE_DIR}/ProyectoFinalIA.v5i.yolov8.zip',
    '/content/drive/MyDrive/dataset.zip',
    '/content/drive/MyDrive/ProyectoFinalIA.v5i.yolov8.zip'
]
candidatos_zip += glob.glob('/content/*.zip')
if os.path.exists(DRIVE_DIR):
    candidatos_zip += glob.glob(f'{DRIVE_DIR}/*.zip')

zip_encontrado = None
for z in candidatos_zip:
    if os.path.exists(z) and os.path.getsize(z) > 1000000:
        zip_encontrado = z
        break

if not zip_encontrado:
    print("⚠️ No se encontró dataset.zip automáticamente. Súbelo ahora:")
    from google.colab import files
    subidos = files.upload()
    for nom in subidos.keys():
        if nom.endswith('.zip'):
            zip_encontrado = nom
            break

print(f"📦 Descomprimiendo y normalizando rutas de: {zip_encontrado}")
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
os.makedirs(DATASET_DIR, exist_ok=True)

# Extraer convirtiendo barras invertidas de Windows '\' a barras de Linux '/'
with zipfile.ZipFile(zip_encontrado, 'r') as zip_ref:
    for member in zip_ref.namelist():
        clean_path = member.replace('\\', '/').lstrip('/')
        target_path = os.path.join(DATASET_DIR, clean_path)
        if member.endswith('/') or member.endswith('\\'):
            os.makedirs(target_path, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target_path), exist_ok=True)
            with zip_ref.open(member) as source, open(target_path, 'wb') as target:
                shutil.copyfileobj(source, target)

print("✅ Dataset descomprimido con carpetas normalizadas para Linux.")

# 2. Localizar inteligentemente las carpetas con imágenes
train_img_dir = None
val_img_dir = None
test_img_dir = None

for root, dirs, files_in_dir in os.walk(DATASET_DIR):
    norm_root = root.replace('\\', '/')
    imgs = [f for f in files_in_dir if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        if 'train' in norm_root:
            train_img_dir = norm_root
        elif 'valid' in norm_root or 'val' in norm_root:
            val_img_dir = norm_root
        elif 'test' in norm_root:
            test_img_dir = norm_root

print("\n--- 🔍 Rutas Detectadas ---")
print(f"📁 Train: {train_img_dir}")
print(f"📁 Val:   {val_img_dir}")
print(f"📁 Test:  {test_img_dir}")

if not train_img_dir or not val_img_dir:
    raise FileNotFoundError("❌ No se pudieron detectar las carpetas de imágenes. Verifica que el archivo zip contenga las fotos.")

n_train = len([f for f in os.listdir(train_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
n_val = len([f for f in os.listdir(val_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f"🖼️ Total imágenes listas -> Entrenamiento: {n_train} | Validación: {n_val}")

# 3. Escribir data.yaml con RUTAS ABSOLUTAS exactas
data_yaml_content = {
    'train': train_img_dir,
    'val': val_img_dir,
    'nc': 6,
    'names': ['Aceite', 'Atun', 'Leche', 'Refresco', 'Sopa', 'Yogurt']
}
if test_img_dir:
    data_yaml_content['test'] = test_img_dir

YAML_FILE = '/content/dataset/data.yaml'
with open(YAML_FILE, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f"\n✅ {YAML_FILE} creado exitosamente con rutas absolutas:")
print(open(YAML_FILE).read())

## 3️⃣ Instalar Ultralytics y Verificar GPU

In [ ]:
%pip install ultralytics onnx onnxslim
import torch
import ultralytics

ultralytics.checks()
if torch.cuda.is_available():
    print(f"🚀 GPU Lista para acelerar: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No se detectó GPU. Ve a Entorno de ejecución -> Cambiar tipo de entorno y selecciona GPU T4.")

## 4️⃣ Entrenamiento Optimizado (YOLOv8m Transfer Learning)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=40,
    patience=12,
    batch=16,
    imgsz=640,
    device=0,
    optimizer='auto',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    mosaic=1.0,
    mixup=0.15,
    degrees=10.0,
    flipud=0.1,
    fliplr=0.5,
    project=f'{DRIVE_DIR}/entrenamientos',
    name='tiendita_yolo_final',
    exist_ok=True,
    save=True,
    verbose=True
)

## 5️⃣ Evaluación de Métricas de Precisión (mAP)

In [ ]:
best_pt_path = f'{DRIVE_DIR}/entrenamientos/tiendita_yolo_final/weights/best.pt'
print(f"🎯 Cargando mejor modelo: {best_pt_path}")
trained_model = YOLO(best_pt_path)

metrics = trained_model.val(data='/content/dataset/data.yaml', split='val')
print("\n" + "="*60)
print(f"🏆 Precisión General (Precision): {metrics.box.mp:.4f}")
print(f"🏆 Cobertura (Recall):            {metrics.box.mr:.4f}")
print(f"🏆 mAP@50:                         {metrics.box.map50:.4f}")
print(f"🏆 mAP@50-95:                      {metrics.box.map:.4f}")
print("="*60)

## 6️⃣ Limpieza de Metadatos y Exportación a ONNX (Opset 12)

In [ ]:
import os
import shutil
import torch
import onnx
from ultralytics import YOLO

FINAL_PT = f'{DRIVE_DIR}/modelos_finales/MiModelo_YOLO_BEST.pt'
FINAL_ONNX = f'{DRIVE_DIR}/modelos_finales/MiModelo_YOLO_BEST.onnx'

os.makedirs(os.path.dirname(FINAL_PT), exist_ok=True)
shutil.copy2(best_pt_path, FINAL_PT)

# Limpieza de metadatos .pt
print("🧹 Limpiando metadatos de PyTorch...")
ckpt = torch.load(FINAL_PT, map_location='cpu', weights_only=False)
for k in ['date', 'license', 'docs', 'git', 'train_args', 'train_metrics', 'train_results']:
    if k in ckpt:
        del ckpt[k]
if hasattr(ckpt.get('model'), 'args'):
    ckpt['model'].args = None
torch.save(ckpt, FINAL_PT)
print(f"✅ Modelo PyTorch limpio: {FINAL_PT}")

# Exportar a ONNX
print("🔄 Exportando a ONNX optimizado (opset 12)...!")
limpio_model = YOLO(FINAL_PT)
onnx_exportado = str(limpio_model.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True,
    dynamic=False
))

# Evitar SameFileError si YOLO ya lo guardó en la ruta final
if os.path.exists(onnx_exportado) and os.path.abspath(onnx_exportado) != os.path.abspath(FINAL_ONNX):
    shutil.copy2(onnx_exportado, FINAL_ONNX)

# Limpieza de metadatos ONNX
print("🧹 Limpiando metadatos de ONNX...")
m_onnx = onnx.load(FINAL_ONNX)
keep_keys = {'names', 'imgsz', 'stride', 'task'}
new_props = [p for p in m_onnx.metadata_props if p.key in keep_keys]
del m_onnx.metadata_props[:]
m_onnx.metadata_props.extend(new_props)
m_onnx.doc_string = ''
m_onnx.producer_name = 'ONNX'
m_onnx.producer_version = '1.0'
onnx.save(m_onnx, FINAL_ONNX)
print(f"✅ Modelo ONNX limpio y optimizado: {FINAL_ONNX}")

print("\n" + "="*70)
print("🎉 ¡MODELOS GENERADOS CON ÉXITO EN TU GOOGLE DRIVE!")
print(f"1. {FINAL_PT}")
print(f"2. {FINAL_ONNX}")
print("="*70)


## 7️⃣ Descarga Directa a tu Computadora

In [ ]:
from google.colab import files

print("⬇️ Descargando MiModelo_YOLO_BEST.onnx...")
files.download(FINAL_ONNX)

print("⬇️ Descargando MiModelo_YOLO_BEST.pt...")
files.download(FINAL_PT)